# 04 — Build Executive KPI Snapshot

## Purpose

Create a consolidated executive KPI snapshot from the validated Gold analytical models.

The dataset summarizes customer health, recurring revenue, invoice exposure, confirmed revenue leakage, and payment recovery performance in a single business-facing record.

## Grain

One record per analytics snapshot date.

## Sources

- `workspace.revenue_leakage_gold.customer_360`
- `workspace.revenue_leakage_gold.revenue_leakage`
- `workspace.revenue_leakage_gold.payment_recovery`

## Target

- `workspace.revenue_leakage_gold.executive_kpis`

## Business Outcomes

- Monitor total customers and customer risk distribution
- Track monthly and annual recurring revenue
- Measure invoiced, collected, outstanding, and past-due revenue
- Quantify confirmed leakage and revenue currently at risk
- Monitor retry recovery performance
- Surface critical collection and recovery workloads
- Provide trusted KPIs for the executive dashboard

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType


CATALOG = "workspace"
GOLD_SCHEMA = "revenue_leakage_gold"

CUSTOMER_360_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.customer_360"
)

REVENUE_LEAKAGE_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.revenue_leakage"
)

PAYMENT_RECOVERY_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.payment_recovery"
)

EXECUTIVE_KPIS_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.executive_kpis"
)


EXPECTED_CUSTOMER_COUNT = 5_150
EXPECTED_REVENUE_LEAKAGE_COUNT = 26_699
EXPECTED_PAYMENT_RECOVERY_COUNT = 25_748


spark.sql(
    f"""
    CREATE SCHEMA IF NOT EXISTS
    {CATALOG}.{GOLD_SCHEMA}
    """
)


customer_360_df = spark.table(
    CUSTOMER_360_TABLE
)

revenue_leakage_df = spark.table(
    REVENUE_LEAKAGE_TABLE
)

payment_recovery_df = spark.table(
    PAYMENT_RECOVERY_TABLE
)


required_customer_columns = {
    "customer_id",
    "customer_status",
    "risk_tier",
    "customer_value_tier",
    "current_mrr",
    "current_arr",
    "analytics_snapshot_date",
}

required_revenue_leakage_columns = {
    "customer_id",
    "invoice_id",
    "invoice_status",
    "invoice_total_amount",
    "amount_paid",
    "outstanding_amount",
    "voided_amount",
    "confirmed_leakage_amount",
    "revenue_at_risk_amount",
    "total_revenue_exposure_amount",
    "recovered_revenue_amount",
    "leakage_status",
    "collection_priority",
    "analytics_snapshot_date",
}

required_payment_recovery_columns = {
    "invoice_id",
    "payment_attempt_count",
    "first_attempt_success_flag",
    "recovery_eligible_flag",
    "recovery_success_flag",
    "unrecovered_failure_flag",
    "pending_collection_flag",
    "recovered_amount",
    "unrecovered_amount",
    "pending_collection_amount",
    "recovery_status",
    "recovery_priority",
    "analytics_snapshot_date",
}


missing_customer_columns = (
    required_customer_columns
    - set(customer_360_df.columns)
)

missing_revenue_leakage_columns = (
    required_revenue_leakage_columns
    - set(revenue_leakage_df.columns)
)

missing_payment_recovery_columns = (
    required_payment_recovery_columns
    - set(payment_recovery_df.columns)
)


assert not missing_customer_columns, (
    "Missing Customer 360 columns: "
    f"{sorted(missing_customer_columns)}"
)

assert not missing_revenue_leakage_columns, (
    "Missing Revenue Leakage columns: "
    f"{sorted(missing_revenue_leakage_columns)}"
)

assert not missing_payment_recovery_columns, (
    "Missing Payment Recovery columns: "
    f"{sorted(missing_payment_recovery_columns)}"
)


customer_360_count = customer_360_df.count()

distinct_customer_count = (
    customer_360_df
    .select("customer_id")
    .distinct()
    .count()
)

revenue_leakage_count = revenue_leakage_df.count()

distinct_revenue_leakage_invoice_count = (
    revenue_leakage_df
    .select("invoice_id")
    .distinct()
    .count()
)

payment_recovery_count = payment_recovery_df.count()

distinct_payment_recovery_invoice_count = (
    payment_recovery_df
    .select("invoice_id")
    .distinct()
    .count()
)


revenue_leakage_customer_reference_errors = (
    revenue_leakage_df
    .select("customer_id")
    .distinct()
    .join(
        customer_360_df.select("customer_id"),
        on="customer_id",
        how="left_anti",
    )
    .count()
)

payment_recovery_invoice_reference_errors = (
    payment_recovery_df
    .select("invoice_id")
    .distinct()
    .join(
        revenue_leakage_df.select("invoice_id"),
        on="invoice_id",
        how="left_anti",
    )
    .count()
)


source_snapshot_dates_df = (
    customer_360_df
    .select(
        F.lit("customer_360").alias("source_name"),
        F.to_date("analytics_snapshot_date").alias(
            "analytics_snapshot_date"
        ),
    )
    .unionByName(
        revenue_leakage_df.select(
            F.lit("revenue_leakage").alias("source_name"),
            F.to_date("analytics_snapshot_date").alias(
                "analytics_snapshot_date"
            ),
        )
    )
    .unionByName(
        payment_recovery_df.select(
            F.lit("payment_recovery").alias("source_name"),
            F.to_date("analytics_snapshot_date").alias(
                "analytics_snapshot_date"
            ),
        )
    )
)


source_snapshot_summary_df = (
    source_snapshot_dates_df
    .groupBy("source_name")
    .agg(
        F.countDistinct(
            "analytics_snapshot_date"
        ).alias("distinct_snapshot_date_count"),
        F.min(
            "analytics_snapshot_date"
        ).alias("minimum_snapshot_date"),
        F.max(
            "analytics_snapshot_date"
        ).alias("maximum_snapshot_date"),
    )
    .orderBy("source_name")
)


analytics_snapshot_dates = [
    row["analytics_snapshot_date"]
    for row in (
        source_snapshot_dates_df
        .select("analytics_snapshot_date")
        .where(
            F.col("analytics_snapshot_date").isNotNull()
        )
        .distinct()
        .collect()
    )
]


assert customer_360_count == EXPECTED_CUSTOMER_COUNT, (
    "Unexpected Customer 360 count: "
    f"{customer_360_count:,}"
)

assert distinct_customer_count == EXPECTED_CUSTOMER_COUNT, (
    "Unexpected distinct Customer 360 count: "
    f"{distinct_customer_count:,}"
)

assert revenue_leakage_count == EXPECTED_REVENUE_LEAKAGE_COUNT, (
    "Unexpected Revenue Leakage count: "
    f"{revenue_leakage_count:,}"
)

assert (
    distinct_revenue_leakage_invoice_count
    == EXPECTED_REVENUE_LEAKAGE_COUNT
), (
    "Unexpected distinct Revenue Leakage invoice count: "
    f"{distinct_revenue_leakage_invoice_count:,}"
)

assert payment_recovery_count == EXPECTED_PAYMENT_RECOVERY_COUNT, (
    "Unexpected Payment Recovery count: "
    f"{payment_recovery_count:,}"
)

assert (
    distinct_payment_recovery_invoice_count
    == EXPECTED_PAYMENT_RECOVERY_COUNT
), (
    "Unexpected distinct Payment Recovery invoice count: "
    f"{distinct_payment_recovery_invoice_count:,}"
)

assert revenue_leakage_customer_reference_errors == 0, (
    "Revenue Leakage customer reference errors detected."
)

assert payment_recovery_invoice_reference_errors == 0, (
    "Payment Recovery invoice reference errors detected."
)

assert len(analytics_snapshot_dates) == 1, (
    "Gold sources do not share one analytics snapshot date: "
    f"{analytics_snapshot_dates}"
)


analytics_snapshot_date = analytics_snapshot_dates[0]


print(
    "Customer 360 records: "
    f"{customer_360_count:,}"
)

print(
    "Distinct Customer IDs: "
    f"{distinct_customer_count:,}"
)

print(
    "Revenue Leakage records: "
    f"{revenue_leakage_count:,}"
)

print(
    "Distinct Revenue Leakage invoice IDs: "
    f"{distinct_revenue_leakage_invoice_count:,}"
)

print(
    "Payment Recovery records: "
    f"{payment_recovery_count:,}"
)

print(
    "Distinct Payment Recovery invoice IDs: "
    f"{distinct_payment_recovery_invoice_count:,}"
)

print(
    "Revenue Leakage customer reference errors: "
    f"{revenue_leakage_customer_reference_errors:,}"
)

print(
    "Payment Recovery invoice reference errors: "
    f"{payment_recovery_invoice_reference_errors:,}"
)

print(
    "Analytics snapshot date: "
    f"{analytics_snapshot_date}"
)

print(
    "All required Gold analytical sources are available."
)


display(
    source_snapshot_summary_df
)

## 2. Build Customer Executive KPIs

Aggregate the Customer 360 model into executive-level customer health, value, recurring revenue, and subscription coverage metrics.

The resulting dataset contains one record for the analytics snapshot date and preserves the exact customer, risk, value, MRR, and ARR totals from Customer 360.

In [0]:
# Aggregate Customer 360 into one executive KPI record.

customer_kpis_base_df = (
    customer_360_df
    .agg(
        F.count("*").alias(
            "total_customer_count"
        ),
        F.countDistinct("customer_id").alias(
            "distinct_customer_count"
        ),
        F.sum(
            F.when(
                F.col("customer_status") == "Active",
                1,
            ).otherwise(0)
        ).alias(
            "active_customer_count"
        ),
        F.sum(
            F.when(
                F.col("customer_status") == "Inactive",
                1,
            ).otherwise(0)
        ).alias(
            "inactive_customer_count"
        ),
        F.sum(
            F.when(
                F.col("risk_tier") == "High",
                1,
            ).otherwise(0)
        ).alias(
            "high_risk_customer_count"
        ),
        F.sum(
            F.when(
                F.col("risk_tier") == "Medium",
                1,
            ).otherwise(0)
        ).alias(
            "medium_risk_customer_count"
        ),
        F.sum(
            F.when(
                F.col("risk_tier") == "Low",
                1,
            ).otherwise(0)
        ).alias(
            "low_risk_customer_count"
        ),
        F.sum(
            F.when(
                F.col("customer_value_tier") == "High Value",
                1,
            ).otherwise(0)
        ).alias(
            "high_value_customer_count"
        ),
        F.sum(
            F.when(
                F.col("customer_value_tier") == "Growth Value",
                1,
            ).otherwise(0)
        ).alias(
            "growth_value_customer_count"
        ),
        F.sum(
            F.when(
                F.col("customer_value_tier") == "Standard Value",
                1,
            ).otherwise(0)
        ).alias(
            "standard_value_customer_count"
        ),
        F.sum(
            F.when(
                F.col("active_subscription_count") > 0,
                1,
            ).otherwise(0)
        ).alias(
            "customers_with_active_subscription_count"
        ),
        F.coalesce(
            F.sum("active_subscription_count"),
            F.lit(0),
        ).cast("long").alias(
            "active_subscription_count"
        ),
        F.round(
            F.coalesce(
                F.sum("current_mrr"),
                F.lit(0),
            ),
            2,
        ).alias(
            "current_mrr"
        ),
        F.round(
            F.coalesce(
                F.sum("current_arr"),
                F.lit(0),
            ),
            2,
        ).alias(
            "current_arr"
        ),
    )
)


customer_kpis_df = (
    customer_kpis_base_df
    .withColumn(
        "analytics_snapshot_date",
        F.lit(analytics_snapshot_date).cast("date"),
    )
    .withColumn(
        "active_customer_rate",
        F.round(
            (
                F.col("active_customer_count")
                / F.col("total_customer_count")
            ) * 100,
            2,
        ),
    )
    .withColumn(
        "high_risk_customer_rate",
        F.round(
            (
                F.col("high_risk_customer_count")
                / F.col("total_customer_count")
            ) * 100,
            2,
        ),
    )
    .withColumn(
        "active_subscription_coverage_rate",
        F.round(
            (
                F.col(
                    "customers_with_active_subscription_count"
                )
                / F.col("total_customer_count")
            ) * 100,
            2,
        ),
    )
    .withColumn(
        "average_mrr_per_customer",
        F.round(
            F.col("current_mrr")
            / F.col("total_customer_count"),
            2,
        ),
    )
    .select(
        "analytics_snapshot_date",
        "total_customer_count",
        "distinct_customer_count",
        "active_customer_count",
        "inactive_customer_count",
        "active_customer_rate",
        "high_risk_customer_count",
        "medium_risk_customer_count",
        "low_risk_customer_count",
        "high_risk_customer_rate",
        "high_value_customer_count",
        "growth_value_customer_count",
        "standard_value_customer_count",
        "customers_with_active_subscription_count",
        "active_subscription_count",
        "active_subscription_coverage_rate",
        "current_mrr",
        "current_arr",
        "average_mrr_per_customer",
    )
)


customer_kpi_row = customer_kpis_df.first()


assert (
    customer_kpi_row["total_customer_count"]
    == EXPECTED_CUSTOMER_COUNT
), "Unexpected executive customer count."

assert (
    customer_kpi_row["distinct_customer_count"]
    == EXPECTED_CUSTOMER_COUNT
), "Unexpected distinct executive customer count."

assert (
    customer_kpi_row["active_customer_count"]
    + customer_kpi_row["inactive_customer_count"]
    == EXPECTED_CUSTOMER_COUNT
), "Customer status counts do not reconcile."

assert (
    customer_kpi_row["high_risk_customer_count"]
    + customer_kpi_row["medium_risk_customer_count"]
    + customer_kpi_row["low_risk_customer_count"]
    == EXPECTED_CUSTOMER_COUNT
), "Customer risk counts do not reconcile."

assert (
    customer_kpi_row["high_value_customer_count"]
    + customer_kpi_row["growth_value_customer_count"]
    + customer_kpi_row["standard_value_customer_count"]
    == EXPECTED_CUSTOMER_COUNT
), "Customer value-tier counts do not reconcile."

assert abs(
    float(customer_kpi_row["current_mrr"])
    - 450_435.70
) < 0.01, "Customer MRR does not reconcile."

assert abs(
    float(customer_kpi_row["current_arr"])
    - 5_405_228.40
) < 0.01, "Customer ARR does not reconcile."

assert abs(
    float(customer_kpi_row["current_arr"])
    - (
        float(customer_kpi_row["current_mrr"])
        * 12
    )
) < 0.01, "Customer MRR and ARR do not reconcile."


print(
    "Total customers: "
    f"{customer_kpi_row['total_customer_count']:,}"
)

print(
    "Active customers: "
    f"{customer_kpi_row['active_customer_count']:,}"
)

print(
    "Inactive customers: "
    f"{customer_kpi_row['inactive_customer_count']:,}"
)

print(
    "High-risk customers: "
    f"{customer_kpi_row['high_risk_customer_count']:,}"
)

print(
    "Customers with active subscriptions: "
    f"{customer_kpi_row['customers_with_active_subscription_count']:,}"
)

print(
    "Active subscriptions: "
    f"{customer_kpi_row['active_subscription_count']:,}"
)

print(
    "Current MRR: "
    f"{float(customer_kpi_row['current_mrr']):,.2f}"
)

print(
    "Current ARR: "
    f"{float(customer_kpi_row['current_arr']):,.2f}"
)

print(
    "Customer executive KPIs reconciled successfully."
)


display(
    customer_kpis_df
)

## 3. Build Revenue and Leakage Executive KPIs

Aggregate the Revenue Leakage model into executive-level billing, collection, outstanding balance, confirmed leakage, revenue-at-risk, recovery, and collection-priority metrics.

All financial totals reconcile with the validated Silver invoice balances and the Gold Revenue Leakage model.

In [0]:
# Aggregate Revenue Leakage into one executive KPI record.

revenue_kpis_base_df = (
    revenue_leakage_df
    .agg(
        F.count("*").alias(
            "total_invoice_count"
        ),
        F.countDistinct("invoice_id").alias(
            "distinct_invoice_count"
        ),
        F.sum(
            F.when(
                F.col("invoice_status") == "Open",
                1,
            ).otherwise(0)
        ).alias(
            "open_invoice_count"
        ),
        F.sum(
            F.when(
                F.col("invoice_status") == "Paid",
                1,
            ).otherwise(0)
        ).alias(
            "paid_invoice_count"
        ),
        F.sum(
            F.when(
                F.col("invoice_status") == "Past Due",
                1,
            ).otherwise(0)
        ).alias(
            "past_due_invoice_count"
        ),
        F.sum(
            F.when(
                F.col("invoice_status") == "Voided",
                1,
            ).otherwise(0)
        ).alias(
            "voided_invoice_count"
        ),
        F.sum(
            F.when(
                F.col("leakage_status") == "Confirmed",
                1,
            ).otherwise(0)
        ).alias(
            "confirmed_leakage_invoice_count"
        ),
        F.sum(
            F.when(
                F.col("leakage_status") == "At Risk",
                1,
            ).otherwise(0)
        ).alias(
            "revenue_at_risk_invoice_count"
        ),
        F.sum(
            F.when(
                F.col("leakage_status") == "Recovered",
                1,
            ).otherwise(0)
        ).alias(
            "recovered_revenue_invoice_count"
        ),
        F.sum(
            F.when(
                F.col("leakage_status") == "No Exposure",
                1,
            ).otherwise(0)
        ).alias(
            "no_exposure_invoice_count"
        ),
        F.sum(
            F.when(
                F.col("leakage_status") == "Excluded",
                1,
            ).otherwise(0)
        ).alias(
            "excluded_invoice_count"
        ),
        F.sum(
            F.when(
                F.col("collection_priority") == "Critical",
                1,
            ).otherwise(0)
        ).alias(
            "critical_collection_invoice_count"
        ),
        F.sum(
            F.when(
                F.col("collection_priority") == "High",
                1,
            ).otherwise(0)
        ).alias(
            "high_collection_invoice_count"
        ),
        F.sum(
            F.when(
                F.col("collection_priority") == "Medium",
                1,
            ).otherwise(0)
        ).alias(
            "medium_collection_invoice_count"
        ),
        F.sum(
            F.when(
                F.col("collection_priority") == "Low",
                1,
            ).otherwise(0)
        ).alias(
            "low_collection_invoice_count"
        ),
        F.round(
            F.coalesce(
                F.sum("invoice_total_amount"),
                F.lit(0),
            ),
            2,
        ).alias(
            "gross_invoiced_amount"
        ),
        F.round(
            F.coalesce(
                F.sum("amount_paid"),
                F.lit(0),
            ),
            2,
        ).alias(
            "collected_amount"
        ),
        F.round(
            F.coalesce(
                F.sum("outstanding_amount"),
                F.lit(0),
            ),
            2,
        ).alias(
            "outstanding_amount"
        ),
        F.round(
            F.coalesce(
                F.sum("voided_amount"),
                F.lit(0),
            ),
            2,
        ).alias(
            "voided_amount"
        ),
        F.round(
            F.coalesce(
                F.sum("confirmed_leakage_amount"),
                F.lit(0),
            ),
            2,
        ).alias(
            "confirmed_leakage_amount"
        ),
        F.round(
            F.coalesce(
                F.sum("revenue_at_risk_amount"),
                F.lit(0),
            ),
            2,
        ).alias(
            "revenue_at_risk_amount"
        ),
        F.round(
            F.coalesce(
                F.sum("total_revenue_exposure_amount"),
                F.lit(0),
            ),
            2,
        ).alias(
            "total_revenue_exposure_amount"
        ),
        F.round(
            F.coalesce(
                F.sum("recovered_revenue_amount"),
                F.lit(0),
            ),
            2,
        ).alias(
            "recovered_revenue_amount"
        ),
    )
)


revenue_kpis_df = (
    revenue_kpis_base_df
    .withColumn(
        "analytics_snapshot_date",
        F.lit(analytics_snapshot_date).cast("date"),
    )
    .withColumn(
        "collectible_revenue_amount",
        F.round(
            F.col("gross_invoiced_amount")
            - F.col("voided_amount"),
            2,
        ),
    )
    .withColumn(
        "collection_rate",
        F.round(
            (
                F.col("collected_amount")
                / F.col("collectible_revenue_amount")
            ) * 100,
            2,
        ),
    )
    .withColumn(
        "confirmed_leakage_rate",
        F.round(
            (
                F.col("confirmed_leakage_amount")
                / F.col("collectible_revenue_amount")
            ) * 100,
            2,
        ),
    )
    .withColumn(
        "revenue_at_risk_rate",
        F.round(
            (
                F.col("revenue_at_risk_amount")
                / F.col("collectible_revenue_amount")
            ) * 100,
            2,
        ),
    )
    .withColumn(
        "total_revenue_exposure_rate",
        F.round(
            (
                F.col("total_revenue_exposure_amount")
                / F.col("collectible_revenue_amount")
            ) * 100,
            2,
        ),
    )
    .withColumn(
        "average_invoice_amount",
        F.round(
            F.col("gross_invoiced_amount")
            / F.col("total_invoice_count"),
            2,
        ),
    )
    .select(
        "analytics_snapshot_date",
        "total_invoice_count",
        "distinct_invoice_count",
        "open_invoice_count",
        "paid_invoice_count",
        "past_due_invoice_count",
        "voided_invoice_count",
        "confirmed_leakage_invoice_count",
        "revenue_at_risk_invoice_count",
        "recovered_revenue_invoice_count",
        "no_exposure_invoice_count",
        "excluded_invoice_count",
        "critical_collection_invoice_count",
        "high_collection_invoice_count",
        "medium_collection_invoice_count",
        "low_collection_invoice_count",
        "gross_invoiced_amount",
        "voided_amount",
        "collectible_revenue_amount",
        "collected_amount",
        "outstanding_amount",
        "confirmed_leakage_amount",
        "revenue_at_risk_amount",
        "total_revenue_exposure_amount",
        "recovered_revenue_amount",
        "collection_rate",
        "confirmed_leakage_rate",
        "revenue_at_risk_rate",
        "total_revenue_exposure_rate",
        "average_invoice_amount",
    )
)


revenue_kpi_row = revenue_kpis_df.first()


assert (
    revenue_kpi_row["total_invoice_count"]
    == EXPECTED_REVENUE_LEAKAGE_COUNT
), "Unexpected executive invoice count."

assert (
    revenue_kpi_row["distinct_invoice_count"]
    == EXPECTED_REVENUE_LEAKAGE_COUNT
), "Unexpected distinct executive invoice count."

assert (
    revenue_kpi_row["open_invoice_count"]
    + revenue_kpi_row["paid_invoice_count"]
    + revenue_kpi_row["past_due_invoice_count"]
    + revenue_kpi_row["voided_invoice_count"]
    == EXPECTED_REVENUE_LEAKAGE_COUNT
), "Invoice status counts do not reconcile."

assert (
    revenue_kpi_row["confirmed_leakage_invoice_count"]
    + revenue_kpi_row["revenue_at_risk_invoice_count"]
    + revenue_kpi_row["recovered_revenue_invoice_count"]
    + revenue_kpi_row["no_exposure_invoice_count"]
    + revenue_kpi_row["excluded_invoice_count"]
    == EXPECTED_REVENUE_LEAKAGE_COUNT
), "Revenue Leakage status counts do not reconcile."


expected_revenue_amounts = {
    "gross_invoiced_amount": 4_615_670.55,
    "collected_amount": 3_806_266.54,
    "outstanding_amount": 729_286.01,
    "voided_amount": 80_118.00,
    "confirmed_leakage_amount": 646_033.53,
    "revenue_at_risk_amount": 83_252.48,
    "total_revenue_exposure_amount": 729_286.01,
    "recovered_revenue_amount": 583_858.90,
}


for metric_name, expected_value in expected_revenue_amounts.items():
    actual_value = float(
        revenue_kpi_row[metric_name]
    )

    assert abs(
        actual_value - expected_value
    ) < 0.01, (
        f"{metric_name} does not reconcile. "
        f"Expected={expected_value:,.2f}, "
        f"Actual={actual_value:,.2f}"
    )


invoice_balance_difference = round(
    float(revenue_kpi_row["gross_invoiced_amount"])
    - (
        float(revenue_kpi_row["collected_amount"])
        + float(revenue_kpi_row["outstanding_amount"])
        + float(revenue_kpi_row["voided_amount"])
    ),
    2,
)

revenue_exposure_difference = round(
    float(
        revenue_kpi_row[
            "total_revenue_exposure_amount"
        ]
    )
    - (
        float(
            revenue_kpi_row[
                "confirmed_leakage_amount"
            ]
        )
        + float(
            revenue_kpi_row[
                "revenue_at_risk_amount"
            ]
        )
    ),
    2,
)


assert invoice_balance_difference == 0.00, (
    "Executive invoice balance does not reconcile."
)

assert revenue_exposure_difference == 0.00, (
    "Executive revenue exposure does not reconcile."
)


print(
    "Total invoices: "
    f"{revenue_kpi_row['total_invoice_count']:,}"
)

print(
    "Paid invoices: "
    f"{revenue_kpi_row['paid_invoice_count']:,}"
)

print(
    "Past-due invoices: "
    f"{revenue_kpi_row['past_due_invoice_count']:,}"
)

print(
    "Gross invoiced amount: "
    f"{float(revenue_kpi_row['gross_invoiced_amount']):,.2f}"
)

print(
    "Collected amount: "
    f"{float(revenue_kpi_row['collected_amount']):,.2f}"
)

print(
    "Outstanding amount: "
    f"{float(revenue_kpi_row['outstanding_amount']):,.2f}"
)

print(
    "Confirmed leakage amount: "
    f"{float(revenue_kpi_row['confirmed_leakage_amount']):,.2f}"
)

print(
    "Revenue at risk amount: "
    f"{float(revenue_kpi_row['revenue_at_risk_amount']):,.2f}"
)

print(
    "Recovered revenue amount: "
    f"{float(revenue_kpi_row['recovered_revenue_amount']):,.2f}"
)

print(
    "Collection rate: "
    f"{float(revenue_kpi_row['collection_rate']):,.2f}%"
)

print(
    "Revenue executive KPIs reconciled successfully."
)


display(
    revenue_kpis_df
)

## 4. Build Payment Recovery Executive KPIs

Aggregate the Payment Recovery model into executive-level payment journey, first-attempt success, retry recovery, unrecovered balance, pending collection, and recovery-priority metrics.

The resulting KPIs measure both the operational effectiveness and financial impact of the payment recovery process.

In [0]:
# Aggregate Payment Recovery into one executive KPI record.

payment_recovery_kpis_base_df = (
    payment_recovery_df
    .agg(
        F.count("*").alias(
            "total_payment_journey_count"
        ),
        F.countDistinct("invoice_id").alias(
            "distinct_payment_journey_count"
        ),
        F.coalesce(
            F.sum("payment_attempt_count"),
            F.lit(0),
        ).cast("long").alias(
            "payment_attempt_count"
        ),
        F.coalesce(
            F.sum(
                F.col(
                    "first_attempt_success_flag"
                ).cast("long")
            ),
            F.lit(0),
        ).cast("long").alias(
            "first_attempt_success_count"
        ),
        F.coalesce(
            F.sum(
                F.col(
                    "recovery_eligible_flag"
                ).cast("long")
            ),
            F.lit(0),
        ).cast("long").alias(
            "recovery_eligible_count"
        ),
        F.coalesce(
            F.sum(
                F.col(
                    "recovery_success_flag"
                ).cast("long")
            ),
            F.lit(0),
        ).cast("long").alias(
            "recovery_success_count"
        ),
        F.coalesce(
            F.sum(
                F.col(
                    "unrecovered_failure_flag"
                ).cast("long")
            ),
            F.lit(0),
        ).cast("long").alias(
            "unrecovered_journey_count"
        ),
        F.coalesce(
            F.sum(
                F.col(
                    "pending_collection_flag"
                ).cast("long")
            ),
            F.lit(0),
        ).cast("long").alias(
            "pending_collection_count"
        ),
        F.sum(
            F.when(
                F.col("recovery_status") == "Resolved",
                1,
            ).otherwise(0)
        ).alias(
            "resolved_journey_count"
        ),
        F.sum(
            F.when(
                F.col("recovery_status") == "Recovered",
                1,
            ).otherwise(0)
        ).alias(
            "recovered_journey_count"
        ),
        F.sum(
            F.when(
                F.col("recovery_status") == "Unrecovered",
                1,
            ).otherwise(0)
        ).alias(
            "unrecovered_status_journey_count"
        ),
        F.sum(
            F.when(
                F.col("recovery_status") == "Pending",
                1,
            ).otherwise(0)
        ).alias(
            "pending_status_journey_count"
        ),
        F.sum(
            F.when(
                F.col("recovery_priority") == "Critical",
                1,
            ).otherwise(0)
        ).alias(
            "critical_recovery_journey_count"
        ),
        F.sum(
            F.when(
                F.col("recovery_priority") == "High",
                1,
            ).otherwise(0)
        ).alias(
            "high_recovery_journey_count"
        ),
        F.sum(
            F.when(
                F.col("recovery_priority") == "Medium",
                1,
            ).otherwise(0)
        ).alias(
            "medium_recovery_journey_count"
        ),
        F.round(
            F.coalesce(
                F.sum("payment_attempt_amount"),
                F.lit(0),
            ),
            2,
        ).alias(
            "payment_attempt_amount"
        ),
        F.round(
            F.coalesce(
                F.sum("settled_amount"),
                F.lit(0),
            ),
            2,
        ).alias(
            "settled_amount"
        ),
        F.round(
            F.coalesce(
                F.sum("recovered_amount"),
                F.lit(0),
            ),
            2,
        ).alias(
            "recovered_amount"
        ),
        F.round(
            F.coalesce(
                F.sum("unrecovered_amount"),
                F.lit(0),
            ),
            2,
        ).alias(
            "unrecovered_amount"
        ),
        F.round(
            F.coalesce(
                F.sum("pending_collection_amount"),
                F.lit(0),
            ),
            2,
        ).alias(
            "pending_collection_amount"
        ),
    )
)


payment_recovery_kpis_df = (
    payment_recovery_kpis_base_df
    .withColumn(
        "analytics_snapshot_date",
        F.lit(analytics_snapshot_date).cast("date"),
    )
    .withColumn(
        "first_attempt_success_rate",
        F.round(
            (
                F.col("first_attempt_success_count")
                / F.col("total_payment_journey_count")
            ) * 100,
            2,
        ),
    )
    .withColumn(
        "retry_recovery_rate",
        F.round(
            (
                F.col("recovery_success_count")
                / F.col("recovery_eligible_count")
            ) * 100,
            2,
        ),
    )
    .withColumn(
        "financial_recovery_rate",
        F.round(
            (
                F.col("recovered_amount")
                / (
                    F.col("recovered_amount")
                    + F.col("unrecovered_amount")
                )
            ) * 100,
            2,
        ),
    )
    .withColumn(
        "average_attempts_per_journey",
        F.round(
            F.col("payment_attempt_count")
            / F.col("total_payment_journey_count"),
            2,
        ),
    )
    .select(
        "analytics_snapshot_date",
        "total_payment_journey_count",
        "distinct_payment_journey_count",
        "payment_attempt_count",
        "first_attempt_success_count",
        "first_attempt_success_rate",
        "recovery_eligible_count",
        "recovery_success_count",
        "retry_recovery_rate",
        "unrecovered_journey_count",
        "pending_collection_count",
        "resolved_journey_count",
        "recovered_journey_count",
        "unrecovered_status_journey_count",
        "pending_status_journey_count",
        "critical_recovery_journey_count",
        "high_recovery_journey_count",
        "medium_recovery_journey_count",
        "payment_attempt_amount",
        "settled_amount",
        "recovered_amount",
        "unrecovered_amount",
        "pending_collection_amount",
        "financial_recovery_rate",
        "average_attempts_per_journey",
    )
)


payment_recovery_kpi_row = (
    payment_recovery_kpis_df.first()
)


assert (
    payment_recovery_kpi_row[
        "total_payment_journey_count"
    ]
    == EXPECTED_PAYMENT_RECOVERY_COUNT
), "Unexpected executive Payment Recovery count."

assert (
    payment_recovery_kpi_row[
        "distinct_payment_journey_count"
    ]
    == EXPECTED_PAYMENT_RECOVERY_COUNT
), "Unexpected distinct executive Payment Recovery count."

assert (
    payment_recovery_kpi_row[
        "resolved_journey_count"
    ]
    + payment_recovery_kpi_row[
        "recovered_journey_count"
    ]
    + payment_recovery_kpi_row[
        "unrecovered_status_journey_count"
    ]
    + payment_recovery_kpi_row[
        "pending_status_journey_count"
    ]
    == EXPECTED_PAYMENT_RECOVERY_COUNT
), "Payment Recovery status counts do not reconcile."

assert (
    payment_recovery_kpi_row[
        "recovery_eligible_count"
    ]
    == (
        payment_recovery_kpi_row[
            "recovery_success_count"
        ]
        + payment_recovery_kpi_row[
            "unrecovered_journey_count"
        ]
    )
), "Recovery-eligible journeys do not reconcile."

assert (
    payment_recovery_kpi_row[
        "recovery_success_count"
    ]
    == payment_recovery_kpi_row[
        "recovered_journey_count"
    ]
), "Successful recovery counts do not reconcile."

assert (
    payment_recovery_kpi_row[
        "unrecovered_journey_count"
    ]
    == payment_recovery_kpi_row[
        "unrecovered_status_journey_count"
    ]
), "Unrecovered journey counts do not reconcile."

assert (
    payment_recovery_kpi_row[
        "pending_collection_count"
    ]
    == payment_recovery_kpi_row[
        "pending_status_journey_count"
    ]
), "Pending collection counts do not reconcile."


expected_recovery_counts = {
    "payment_attempt_count": 29_972,
    "first_attempt_success_count": 18_631,
    "recovery_eligible_count": 6_923,
    "recovery_success_count": 3_282,
    "unrecovered_journey_count": 3_641,
    "pending_collection_count": 194,
}


for metric_name, expected_value in expected_recovery_counts.items():
    actual_value = int(
        payment_recovery_kpi_row[metric_name]
    )

    assert actual_value == expected_value, (
        f"{metric_name} does not reconcile. "
        f"Expected={expected_value:,}, "
        f"Actual={actual_value:,}"
    )


expected_recovery_amounts = {
    "payment_attempt_amount": 5_234_901.68,
    "settled_amount": 3_806_266.54,
    "recovered_amount": 583_858.90,
    "unrecovered_amount": 646_033.53,
    "pending_collection_amount": 31_490.34,
}


for metric_name, expected_value in expected_recovery_amounts.items():
    actual_value = float(
        payment_recovery_kpi_row[metric_name]
    )

    assert abs(
        actual_value - expected_value
    ) < 0.01, (
        f"{metric_name} does not reconcile. "
        f"Expected={expected_value:,.2f}, "
        f"Actual={actual_value:,.2f}"
    )


assert abs(
    float(
        payment_recovery_kpi_row[
            "retry_recovery_rate"
        ]
    )
    - 47.41
) < 0.01, "Retry recovery rate does not reconcile."


print(
    "Payment journeys: "
    f"{payment_recovery_kpi_row['total_payment_journey_count']:,}"
)

print(
    "Payment attempts: "
    f"{payment_recovery_kpi_row['payment_attempt_count']:,}"
)

print(
    "First-attempt successes: "
    f"{payment_recovery_kpi_row['first_attempt_success_count']:,}"
)

print(
    "Recovery-eligible journeys: "
    f"{payment_recovery_kpi_row['recovery_eligible_count']:,}"
)

print(
    "Successful recoveries: "
    f"{payment_recovery_kpi_row['recovery_success_count']:,}"
)

print(
    "Unrecovered journeys: "
    f"{payment_recovery_kpi_row['unrecovered_journey_count']:,}"
)

print(
    "Pending collections: "
    f"{payment_recovery_kpi_row['pending_collection_count']:,}"
)

print(
    "Retry recovery rate: "
    f"{float(payment_recovery_kpi_row['retry_recovery_rate']):,.2f}%"
)

print(
    "Recovered amount: "
    f"{float(payment_recovery_kpi_row['recovered_amount']):,.2f}"
)

print(
    "Unrecovered amount: "
    f"{float(payment_recovery_kpi_row['unrecovered_amount']):,.2f}"
)

print(
    "Pending collection amount: "
    f"{float(payment_recovery_kpi_row['pending_collection_amount']):,.2f}"
)

print(
    "Payment Recovery executive KPIs reconciled successfully."
)


display(
    payment_recovery_kpis_df
)

## 5. Assemble the Executive KPI Snapshot

Combine the customer, revenue leakage, and payment recovery KPI datasets into one executive record for the shared analytics snapshot date.

Cross-model reconciliations confirm that collected revenue, settled payments, confirmed leakage, unrecovered payments, and recovered revenue remain consistent across all Gold models.

In [0]:
# Combine all executive KPI domains into one snapshot record.

customer_kpi_columns = [
    column_name
    for column_name in customer_kpis_df.columns
    if column_name != "analytics_snapshot_date"
]

revenue_kpi_columns = [
    column_name
    for column_name in revenue_kpis_df.columns
    if column_name != "analytics_snapshot_date"
]

payment_recovery_kpi_columns = [
    column_name
    for column_name in payment_recovery_kpis_df.columns
    if column_name != "analytics_snapshot_date"
]


executive_kpis_joined_df = (
    customer_kpis_df
    .join(
        revenue_kpis_df,
        on="analytics_snapshot_date",
        how="inner",
    )
    .join(
        payment_recovery_kpis_df,
        on="analytics_snapshot_date",
        how="inner",
    )
)


executive_kpis_df = (
    executive_kpis_joined_df
    .select(
        "analytics_snapshot_date",
        F.date_format(
            "analytics_snapshot_date",
            "yyyyMMdd",
        ).alias(
            "executive_kpi_snapshot_key"
        ),
        F.lit("1.0").alias(
            "kpi_model_version"
        ),
        *customer_kpi_columns,
        *revenue_kpi_columns,
        *payment_recovery_kpi_columns,
    )
    .withColumn(
        "uncollected_revenue_rate",
        F.round(
            F.lit(100)
            - F.col("collection_rate"),
            2,
        ),
    )
    .withColumn(
        "recovery_opportunity_amount",
        F.round(
            F.col("unrecovered_amount")
            + F.col("pending_collection_amount"),
            2,
        ),
    )
    .withColumn(
        "actionable_collection_invoice_count",
        (
            F.col("critical_collection_invoice_count")
            + F.col("high_collection_invoice_count")
            + F.col("medium_collection_invoice_count")
            + F.col("low_collection_invoice_count")
        ).cast("long"),
    )
    .withColumn(
        "actionable_recovery_journey_count",
        (
            F.col("critical_recovery_journey_count")
            + F.col("high_recovery_journey_count")
            + F.col("medium_recovery_journey_count")
        ).cast("long"),
    )
)


executive_kpi_count = executive_kpis_df.count()

distinct_executive_snapshot_count = (
    executive_kpis_df
    .select("executive_kpi_snapshot_key")
    .distinct()
    .count()
)

executive_kpi_row = executive_kpis_df.first()


assert executive_kpi_count == 1, (
    "Expected exactly one executive KPI record."
)

assert distinct_executive_snapshot_count == 1, (
    "Duplicate executive KPI snapshot detected."
)

assert (
    executive_kpi_row["analytics_snapshot_date"]
    == analytics_snapshot_date
), "Unexpected executive analytics snapshot date."

assert (
    executive_kpi_row["total_customer_count"]
    == EXPECTED_CUSTOMER_COUNT
), "Executive customer count does not reconcile."

assert (
    executive_kpi_row["total_invoice_count"]
    == EXPECTED_REVENUE_LEAKAGE_COUNT
), "Executive invoice count does not reconcile."

assert (
    executive_kpi_row["total_payment_journey_count"]
    == EXPECTED_PAYMENT_RECOVERY_COUNT
), "Executive payment journey count does not reconcile."


collected_settlement_difference = round(
    float(executive_kpi_row["collected_amount"])
    - float(executive_kpi_row["settled_amount"]),
    2,
)

leakage_unrecovered_difference = round(
    float(
        executive_kpi_row[
            "confirmed_leakage_amount"
        ]
    )
    - float(
        executive_kpi_row[
            "unrecovered_amount"
        ]
    ),
    2,
)

recovered_revenue_difference = round(
    float(
        executive_kpi_row[
            "recovered_revenue_amount"
        ]
    )
    - float(
        executive_kpi_row[
            "recovered_amount"
        ]
    ),
    2,
)

outstanding_exposure_difference = round(
    float(
        executive_kpi_row[
            "outstanding_amount"
        ]
    )
    - float(
        executive_kpi_row[
            "total_revenue_exposure_amount"
        ]
    ),
    2,
)

recovery_eligible_difference = (
    executive_kpi_row["recovery_eligible_count"]
    - (
        executive_kpi_row[
            "confirmed_leakage_invoice_count"
        ]
        + executive_kpi_row[
            "recovery_success_count"
        ]
    )
)


assert collected_settlement_difference == 0.00, (
    "Collected revenue and settled payments do not reconcile."
)

assert leakage_unrecovered_difference == 0.00, (
    "Confirmed leakage and unrecovered payments do not reconcile."
)

assert recovered_revenue_difference == 0.00, (
    "Recovered revenue values do not reconcile."
)

assert outstanding_exposure_difference == 0.00, (
    "Outstanding balance and revenue exposure do not reconcile."
)

assert recovery_eligible_difference == 0, (
    "Recovery-eligible journey count does not reconcile."
)


print(
    "Executive KPI records: "
    f"{executive_kpi_count:,}"
)

print(
    "Executive snapshot key: "
    f"{executive_kpi_row['executive_kpi_snapshot_key']}"
)

print(
    "Total customers: "
    f"{executive_kpi_row['total_customer_count']:,}"
)

print(
    "Total invoices: "
    f"{executive_kpi_row['total_invoice_count']:,}"
)

print(
    "Payment journeys: "
    f"{executive_kpi_row['total_payment_journey_count']:,}"
)

print(
    "Current MRR: "
    f"{float(executive_kpi_row['current_mrr']):,.2f}"
)

print(
    "Current ARR: "
    f"{float(executive_kpi_row['current_arr']):,.2f}"
)

print(
    "Collected amount: "
    f"{float(executive_kpi_row['collected_amount']):,.2f}"
)

print(
    "Total revenue exposure: "
    f"{float(executive_kpi_row['total_revenue_exposure_amount']):,.2f}"
)

print(
    "Recovered amount: "
    f"{float(executive_kpi_row['recovered_amount']):,.2f}"
)

print(
    "Collection rate: "
    f"{float(executive_kpi_row['collection_rate']):,.2f}%"
)

print(
    "Retry recovery rate: "
    f"{float(executive_kpi_row['retry_recovery_rate']):,.2f}%"
)

print(
    "All executive KPI domains reconciled successfully."
)


executive_kpi_preview_df = (
    executive_kpis_df
    .select(
        "analytics_snapshot_date",
        "total_customer_count",
        "active_customer_count",
        "high_risk_customer_count",
        "current_mrr",
        "current_arr",
        "total_invoice_count",
        "gross_invoiced_amount",
        "collected_amount",
        "outstanding_amount",
        "confirmed_leakage_amount",
        "revenue_at_risk_amount",
        "recovered_amount",
        "collection_rate",
        "retry_recovery_rate",
    )
)


display(
    executive_kpi_preview_df
)

## 6. Validate and Hash the Executive KPI Snapshot

Validate executive record uniqueness, required fields, metric domains, rates, financial balances, and cross-model reconciliations.

Generate a deterministic SHA-256 record hash from all executive business attributes. The hash supports change detection and idempotent Delta merges without creating unnecessary updates.

In [0]:
# Validate the final executive KPI record and generate its deterministic hash.

negative_metric_columns = [
    "total_customer_count",
    "active_customer_count",
    "inactive_customer_count",
    "high_risk_customer_count",
    "medium_risk_customer_count",
    "low_risk_customer_count",
    "active_subscription_count",
    "current_mrr",
    "current_arr",
    "total_invoice_count",
    "gross_invoiced_amount",
    "collected_amount",
    "outstanding_amount",
    "voided_amount",
    "confirmed_leakage_amount",
    "revenue_at_risk_amount",
    "total_revenue_exposure_amount",
    "recovered_revenue_amount",
    "total_payment_journey_count",
    "payment_attempt_count",
    "recovery_eligible_count",
    "recovery_success_count",
    "unrecovered_journey_count",
    "pending_collection_count",
    "payment_attempt_amount",
    "settled_amount",
    "recovered_amount",
    "unrecovered_amount",
    "pending_collection_amount",
    "recovery_opportunity_amount",
]


rate_columns = [
    "active_customer_rate",
    "high_risk_customer_rate",
    "active_subscription_coverage_rate",
    "collection_rate",
    "confirmed_leakage_rate",
    "revenue_at_risk_rate",
    "total_revenue_exposure_rate",
    "uncollected_revenue_rate",
    "first_attempt_success_rate",
    "retry_recovery_rate",
    "financial_recovery_rate",
]


negative_metric_condition = None

for column_name in negative_metric_columns:
    current_condition = (
        F.col(column_name) < 0
    )

    negative_metric_condition = (
        current_condition
        if negative_metric_condition is None
        else negative_metric_condition | current_condition
    )


invalid_rate_condition = None

for column_name in rate_columns:
    current_condition = (
        F.col(column_name).isNull()
        | (F.col(column_name) < 0)
        | (F.col(column_name) > 100)
    )

    invalid_rate_condition = (
        current_condition
        if invalid_rate_condition is None
        else invalid_rate_condition | current_condition
    )


critical_field_condition = (
    F.col("analytics_snapshot_date").isNull()
    | F.col("executive_kpi_snapshot_key").isNull()
    | F.col("kpi_model_version").isNull()
    | F.col("total_customer_count").isNull()
    | F.col("total_invoice_count").isNull()
    | F.col("total_payment_journey_count").isNull()
    | F.col("current_mrr").isNull()
    | F.col("current_arr").isNull()
    | F.col("gross_invoiced_amount").isNull()
    | F.col("collected_amount").isNull()
    | F.col("total_revenue_exposure_amount").isNull()
    | F.col("recovered_amount").isNull()
)


validation_conditions = {
    "null_critical_field_count": (
        critical_field_condition
    ),
    "invalid_customer_count": (
        (F.col("total_customer_count") != EXPECTED_CUSTOMER_COUNT)
        | (
            F.col("distinct_customer_count")
            != EXPECTED_CUSTOMER_COUNT
        )
    ),
    "invalid_invoice_count": (
        (
            F.col("total_invoice_count")
            != EXPECTED_REVENUE_LEAKAGE_COUNT
        )
        | (
            F.col("distinct_invoice_count")
            != EXPECTED_REVENUE_LEAKAGE_COUNT
        )
    ),
    "invalid_payment_journey_count": (
        (
            F.col("total_payment_journey_count")
            != EXPECTED_PAYMENT_RECOVERY_COUNT
        )
        | (
            F.col("distinct_payment_journey_count")
            != EXPECTED_PAYMENT_RECOVERY_COUNT
        )
    ),
    "invalid_negative_metric_count": (
        negative_metric_condition
    ),
    "invalid_rate_count": (
        invalid_rate_condition
    ),
    "invalid_mrr_arr_count": (
        F.abs(
            F.col("current_arr")
            - (
                F.col("current_mrr")
                * F.lit(12)
            )
        ) > F.lit(0.01)
    ),
    "invalid_invoice_balance_count": (
        F.abs(
            F.col("gross_invoiced_amount")
            - (
                F.col("collected_amount")
                + F.col("outstanding_amount")
                + F.col("voided_amount")
            )
        ) > F.lit(0.01)
    ),
    "invalid_revenue_exposure_count": (
        F.abs(
            F.col("total_revenue_exposure_amount")
            - (
                F.col("confirmed_leakage_amount")
                + F.col("revenue_at_risk_amount")
            )
        ) > F.lit(0.01)
    ),
    "invalid_collected_settlement_count": (
        F.abs(
            F.col("collected_amount")
            - F.col("settled_amount")
        ) > F.lit(0.01)
    ),
    "invalid_leakage_unrecovered_count": (
        F.abs(
            F.col("confirmed_leakage_amount")
            - F.col("unrecovered_amount")
        ) > F.lit(0.01)
    ),
    "invalid_recovered_amount_count": (
        F.abs(
            F.col("recovered_revenue_amount")
            - F.col("recovered_amount")
        ) > F.lit(0.01)
    ),
    "invalid_recovery_eligible_count": (
        F.col("recovery_eligible_count")
        != (
            F.col("recovery_success_count")
            + F.col("unrecovered_journey_count")
        )
    ),
}


executive_validation_metrics_df = (
    executive_kpis_df
    .agg(
        *[
            F.sum(
                F.when(
                    validation_condition,
                    1,
                ).otherwise(0)
            ).alias(metric_name)
            for (
                metric_name,
                validation_condition,
            ) in validation_conditions.items()
        ]
    )
)


executive_validation_metrics = (
    executive_validation_metrics_df
    .first()
    .asDict()
)


duplicate_executive_snapshot_count = (
    executive_kpis_df
    .groupBy("executive_kpi_snapshot_key")
    .count()
    .where(
        F.col("count") > 1
    )
    .count()
)


assert duplicate_executive_snapshot_count == 0, (
    "Duplicate executive KPI snapshots detected."
)


for (
    metric_name,
    invalid_record_count,
) in executive_validation_metrics.items():
    assert invalid_record_count == 0, (
        f"Executive KPI validation failed: "
        f"{metric_name}={invalid_record_count}"
    )


executive_hash_columns = [
    column_name
    for column_name in executive_kpis_df.columns
]


executive_kpis_hashed_df = (
    executive_kpis_df
    .withColumn(
        "_record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                *[
                    F.coalesce(
                        F.col(column_name).cast("string"),
                        F.lit("<NULL>"),
                    )
                    for column_name in executive_hash_columns
                ],
            ),
            256,
        ),
    )
)


invalid_record_hash_count = (
    executive_kpis_hashed_df
    .where(
        F.col("_record_hash").isNull()
        | (
            F.length("_record_hash")
            != 64
        )
    )
    .count()
)


assert invalid_record_hash_count == 0, (
    "Invalid executive KPI record hash detected."
)


print(
    "Executive KPI validation results:"
)

for metric_name in sorted(
    executive_validation_metrics.keys()
):
    print(
        f"{metric_name}: "
        f"{executive_validation_metrics[metric_name]:,}"
    )

print(
    "duplicate_executive_snapshot_count: "
    f"{duplicate_executive_snapshot_count:,}"
)

print(
    "invalid_record_hash_count: "
    f"{invalid_record_hash_count:,}"
)

print(
    "Executive KPI validation and hashing "
    "completed successfully."
)


display(
    executive_kpis_hashed_df
    .select(
        "analytics_snapshot_date",
        "executive_kpi_snapshot_key",
        "total_customer_count",
        "current_mrr",
        "current_arr",
        "gross_invoiced_amount",
        "collected_amount",
        "total_revenue_exposure_amount",
        "recovered_amount",
        "collection_rate",
        "retry_recovery_rate",
        "_record_hash",
    )
)

## 7. Persist the Executive KPI Snapshot

Persist the validated executive snapshot as a managed Delta table.

The analytics snapshot key is used as the merge key. Existing records are updated only when the deterministic record hash changes, while identical reruns produce no unnecessary updates.

In [0]:
from delta.tables import DeltaTable


# Persist the executive KPI snapshot using an idempotent Delta MERGE.

executive_business_columns = (
    executive_kpis_hashed_df.columns
)


if not spark.catalog.tableExists(
    EXECUTIVE_KPIS_TABLE
):
    executive_initial_write_df = (
        executive_kpis_hashed_df
        .withColumn(
            "_created_at",
            F.current_timestamp(),
        )
        .withColumn(
            "_updated_at",
            F.current_timestamp(),
        )
    )

    (
        executive_initial_write_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(
            EXECUTIVE_KPIS_TABLE
        )
    )

    executive_write_method = (
        "Initial Delta table creation"
    )

else:
    executive_target_delta = (
        DeltaTable.forName(
            spark,
            EXECUTIVE_KPIS_TABLE,
        )
    )

    executive_update_assignments = {
        column_name: (
            f"source.`{column_name}`"
        )
        for column_name in executive_business_columns
    }

    executive_update_assignments[
        "_updated_at"
    ] = "current_timestamp()"

    executive_insert_assignments = {
        column_name: (
            f"source.`{column_name}`"
        )
        for column_name in executive_business_columns
    }

    executive_insert_assignments[
        "_created_at"
    ] = "current_timestamp()"

    executive_insert_assignments[
        "_updated_at"
    ] = "current_timestamp()"

    (
        executive_target_delta
        .alias("target")
        .merge(
            executive_kpis_hashed_df.alias(
                "source"
            ),
            """
            target.executive_kpi_snapshot_key
            =
            source.executive_kpi_snapshot_key
            """,
        )
        .whenMatchedUpdate(
            condition="""
                target._record_hash
                <>
                source._record_hash
            """,
            set=executive_update_assignments,
        )
        .whenNotMatchedInsert(
            values=executive_insert_assignments,
        )
        .execute()
    )

    executive_write_method = (
        "Delta MERGE"
    )


saved_executive_kpis_df = (
    spark.table(
        EXECUTIVE_KPIS_TABLE
    )
)


saved_current_snapshot_df = (
    saved_executive_kpis_df
    .where(
        F.col(
            "executive_kpi_snapshot_key"
        )
        == executive_kpi_row[
            "executive_kpi_snapshot_key"
        ]
    )
)


saved_current_snapshot_count = (
    saved_current_snapshot_df.count()
)

saved_distinct_snapshot_count = (
    saved_executive_kpis_df
    .select(
        "executive_kpi_snapshot_key"
    )
    .distinct()
    .count()
)


saved_null_critical_field_count = (
    saved_current_snapshot_df
    .where(
        F.col(
            "analytics_snapshot_date"
        ).isNull()
        | F.col(
            "executive_kpi_snapshot_key"
        ).isNull()
        | F.col(
            "_record_hash"
        ).isNull()
    )
    .count()
)


saved_content_mismatch_count = (
    executive_kpis_hashed_df
    .select(
        "executive_kpi_snapshot_key",
        "_record_hash",
    )
    .alias("source")
    .join(
        saved_current_snapshot_df
        .select(
            "executive_kpi_snapshot_key",
            "_record_hash",
        )
        .alias("target"),
        on=(
            F.col(
                "source.executive_kpi_snapshot_key"
            )
            == F.col(
                "target.executive_kpi_snapshot_key"
            )
        ),
        how="left",
    )
    .where(
        F.col(
            "target.executive_kpi_snapshot_key"
        ).isNull()
        | (
            F.col("source._record_hash")
            != F.col("target._record_hash")
        )
    )
    .count()
)


assert saved_current_snapshot_count == 1, (
    "Expected one saved executive KPI snapshot."
)

assert saved_null_critical_field_count == 0, (
    "Saved executive KPI critical fields are null."
)

assert saved_content_mismatch_count == 0, (
    "Saved executive KPI content does not match "
    "the validated source record."
)


saved_executive_kpi_row = (
    saved_current_snapshot_df.first()
)


print(
    "Write method: "
    f"{executive_write_method}"
)

print(
    "Executive KPI table: "
    f"{EXECUTIVE_KPIS_TABLE}"
)

print(
    "Saved current snapshot records: "
    f"{saved_current_snapshot_count:,}"
)

print(
    "Saved distinct analytics snapshots: "
    f"{saved_distinct_snapshot_count:,}"
)

print(
    "Saved content mismatches: "
    f"{saved_content_mismatch_count:,}"
)

print(
    "Saved snapshot key: "
    f"{saved_executive_kpi_row['executive_kpi_snapshot_key']}"
)

print(
    "Saved current MRR: "
    f"{float(saved_executive_kpi_row['current_mrr']):,.2f}"
)

print(
    "Saved collected amount: "
    f"{float(saved_executive_kpi_row['collected_amount']):,.2f}"
)

print(
    "Saved revenue exposure: "
    f"{float(saved_executive_kpi_row['total_revenue_exposure_amount']):,.2f}"
)

print(
    "Executive KPI Delta table persisted successfully."
)


display(
    saved_current_snapshot_df
    .select(
        "analytics_snapshot_date",
        "executive_kpi_snapshot_key",
        "total_customer_count",
        "high_risk_customer_count",
        "current_mrr",
        "current_arr",
        "gross_invoiced_amount",
        "collected_amount",
        "total_revenue_exposure_amount",
        "recovered_amount",
        "collection_rate",
        "retry_recovery_rate",
        "_record_hash",
        "_created_at",
        "_updated_at",
    )
)

## 8. Validate Idempotent Reprocessing

Rerun the Executive KPI Delta merge using the same validated source record.

A successful idempotency test must preserve the existing snapshot without inserting, updating, deleting, or duplicating any records.

In [0]:
# Rerun the Executive KPI merge and verify idempotency.

executive_before_rerun_df = (
    spark.table(
        EXECUTIVE_KPIS_TABLE
    )
)

rows_before_rerun = (
    executive_before_rerun_df.count()
)

snapshot_before_rerun_row = (
    executive_before_rerun_df
    .where(
        F.col(
            "executive_kpi_snapshot_key"
        )
        == executive_kpi_row[
            "executive_kpi_snapshot_key"
        ]
    )
    .first()
)


hash_before_rerun = (
    snapshot_before_rerun_row["_record_hash"]
)

created_at_before_rerun = (
    snapshot_before_rerun_row["_created_at"]
)

updated_at_before_rerun = (
    snapshot_before_rerun_row["_updated_at"]
)


executive_idempotency_target = (
    DeltaTable.forName(
        spark,
        EXECUTIVE_KPIS_TABLE,
    )
)


idempotency_update_assignments = {
    column_name: (
        f"source.`{column_name}`"
    )
    for column_name in executive_business_columns
}

idempotency_update_assignments[
    "_updated_at"
] = "current_timestamp()"


idempotency_insert_assignments = {
    column_name: (
        f"source.`{column_name}`"
    )
    for column_name in executive_business_columns
}

idempotency_insert_assignments[
    "_created_at"
] = "current_timestamp()"

idempotency_insert_assignments[
    "_updated_at"
] = "current_timestamp()"


(
    executive_idempotency_target
    .alias("target")
    .merge(
        executive_kpis_hashed_df.alias(
            "source"
        ),
        """
        target.executive_kpi_snapshot_key
        =
        source.executive_kpi_snapshot_key
        """,
    )
    .whenMatchedUpdate(
        condition="""
            target._record_hash
            <>
            source._record_hash
        """,
        set=idempotency_update_assignments,
    )
    .whenNotMatchedInsert(
        values=idempotency_insert_assignments,
    )
    .execute()
)


executive_after_rerun_df = (
    spark.table(
        EXECUTIVE_KPIS_TABLE
    )
)

rows_after_rerun = (
    executive_after_rerun_df.count()
)


snapshot_after_rerun_row = (
    executive_after_rerun_df
    .where(
        F.col(
            "executive_kpi_snapshot_key"
        )
        == executive_kpi_row[
            "executive_kpi_snapshot_key"
        ]
    )
    .first()
)


hash_after_rerun = (
    snapshot_after_rerun_row["_record_hash"]
)

created_at_after_rerun = (
    snapshot_after_rerun_row["_created_at"]
)

updated_at_after_rerun = (
    snapshot_after_rerun_row["_updated_at"]
)


duplicate_snapshots_after_rerun = (
    executive_after_rerun_df
    .groupBy(
        "executive_kpi_snapshot_key"
    )
    .count()
    .where(
        F.col("count") > 1
    )
    .count()
)


latest_executive_history_df = (
    executive_idempotency_target
    .history(1)
)


latest_executive_history_row = (
    latest_executive_history_df.first()
)

latest_operation_metrics = (
    latest_executive_history_row[
        "operationMetrics"
    ]
    or {}
)


rows_inserted_during_rerun = int(
    latest_operation_metrics.get(
        "numTargetRowsInserted",
        0,
    )
)

rows_updated_during_rerun = int(
    latest_operation_metrics.get(
        "numTargetRowsUpdated",
        0,
    )
)

rows_deleted_during_rerun = int(
    latest_operation_metrics.get(
        "numTargetRowsDeleted",
        0,
    )
)


assert rows_before_rerun == rows_after_rerun, (
    "Executive KPI row count changed during rerun."
)

assert rows_inserted_during_rerun == 0, (
    "Executive KPI rerun inserted records."
)

assert rows_updated_during_rerun == 0, (
    "Executive KPI rerun updated unchanged records."
)

assert rows_deleted_during_rerun == 0, (
    "Executive KPI rerun deleted records."
)

assert duplicate_snapshots_after_rerun == 0, (
    "Executive KPI rerun created duplicate snapshots."
)

assert hash_before_rerun == hash_after_rerun, (
    "Executive KPI record hash changed during rerun."
)

assert (
    created_at_before_rerun
    == created_at_after_rerun
), "Executive KPI creation timestamp changed."

assert (
    updated_at_before_rerun
    == updated_at_after_rerun
), "Unchanged Executive KPI record was updated."


print(
    "Rows before rerun: "
    f"{rows_before_rerun:,}"
)

print(
    "Rows after rerun: "
    f"{rows_after_rerun:,}"
)

print(
    "Rows inserted during rerun: "
    f"{rows_inserted_during_rerun:,}"
)

print(
    "Rows updated during rerun: "
    f"{rows_updated_during_rerun:,}"
)

print(
    "Rows deleted during rerun: "
    f"{rows_deleted_during_rerun:,}"
)

print(
    "Duplicate snapshots after rerun: "
    f"{duplicate_snapshots_after_rerun:,}"
)

print(
    "Record hash unchanged: "
    f"{hash_before_rerun == hash_after_rerun}"
)

print(
    "Creation timestamp unchanged: "
    f"{created_at_before_rerun == created_at_after_rerun}"
)

print(
    "Update timestamp unchanged: "
    f"{updated_at_before_rerun == updated_at_after_rerun}"
)

print(
    "Executive KPI Gold processing is idempotent."
)


display(
    latest_executive_history_df
    .select(
        "version",
        "timestamp",
        "operation",
        "operationMetrics",
    )
)

## 9. Final Executive KPI Output

The Executive KPI Gold model is complete and dashboard-ready.

The model provides one trusted analytical snapshot that reconciles customer health, recurring revenue, invoice collections, revenue leakage, payment recovery, and operational priorities.

All source validations, cross-model financial reconciliations, deterministic hashing, Delta persistence, and idempotency tests completed successfully.

In [0]:
# Present the final dashboard-ready Executive KPI snapshot.

final_executive_kpis_df = (
    spark.table(
        EXECUTIVE_KPIS_TABLE
    )
    .orderBy(
        F.col(
            "analytics_snapshot_date"
        ).desc()
    )
)


final_executive_kpi_count = (
    final_executive_kpis_df.count()
)

final_latest_kpi_row = (
    final_executive_kpis_df.first()
)


assert final_executive_kpi_count >= 1, (
    "The Executive KPI table is empty."
)

assert (
    final_latest_kpi_row[
        "executive_kpi_snapshot_key"
    ]
    == executive_kpi_row[
        "executive_kpi_snapshot_key"
    ]
), "Unexpected latest Executive KPI snapshot."


executive_summary_rows = [
    (
        "Analytics Snapshot Date",
        str(
            final_latest_kpi_row[
                "analytics_snapshot_date"
            ]
        ),
    ),
    (
        "Total Customers",
        f"{final_latest_kpi_row['total_customer_count']:,}",
    ),
    (
        "Active Customers",
        f"{final_latest_kpi_row['active_customer_count']:,}",
    ),
    (
        "High-Risk Customers",
        f"{final_latest_kpi_row['high_risk_customer_count']:,}",
    ),
    (
        "Current MRR",
        f"${float(final_latest_kpi_row['current_mrr']):,.2f}",
    ),
    (
        "Current ARR",
        f"${float(final_latest_kpi_row['current_arr']):,.2f}",
    ),
    (
        "Total Invoices",
        f"{final_latest_kpi_row['total_invoice_count']:,}",
    ),
    (
        "Gross Invoiced Amount",
        f"${float(final_latest_kpi_row['gross_invoiced_amount']):,.2f}",
    ),
    (
        "Collected Amount",
        f"${float(final_latest_kpi_row['collected_amount']):,.2f}",
    ),
    (
        "Collection Rate",
        f"{float(final_latest_kpi_row['collection_rate']):,.2f}%",
    ),
    (
        "Outstanding Revenue",
        f"${float(final_latest_kpi_row['outstanding_amount']):,.2f}",
    ),
    (
        "Confirmed Revenue Leakage",
        f"${float(final_latest_kpi_row['confirmed_leakage_amount']):,.2f}",
    ),
    (
        "Revenue at Risk",
        f"${float(final_latest_kpi_row['revenue_at_risk_amount']):,.2f}",
    ),
    (
        "Payment Journeys",
        f"{final_latest_kpi_row['total_payment_journey_count']:,}",
    ),
    (
        "Recovery-Eligible Journeys",
        f"{final_latest_kpi_row['recovery_eligible_count']:,}",
    ),
    (
        "Successful Recoveries",
        f"{final_latest_kpi_row['recovery_success_count']:,}",
    ),
    (
        "Retry Recovery Rate",
        f"{float(final_latest_kpi_row['retry_recovery_rate']):,.2f}%",
    ),
    (
        "Recovered Revenue",
        f"${float(final_latest_kpi_row['recovered_amount']):,.2f}",
    ),
    (
        "Unrecovered Revenue",
        f"${float(final_latest_kpi_row['unrecovered_amount']):,.2f}",
    ),
    (
        "Pending Collection Amount",
        f"${float(final_latest_kpi_row['pending_collection_amount']):,.2f}",
    ),
]


executive_summary_df = (
    spark.createDataFrame(
        executive_summary_rows,
        [
            "executive_metric",
            "metric_value",
        ],
    )
)


print(
    "Executive KPI Gold model completed successfully."
)

print(
    "Gold table: "
    f"{EXECUTIVE_KPIS_TABLE}"
)

print(
    "Available analytics snapshots: "
    f"{final_executive_kpi_count:,}"
)

print(
    "Latest snapshot key: "
    f"{final_latest_kpi_row['executive_kpi_snapshot_key']}"
)

print(
    "The Executive KPI dataset is ready "
    "for dashboard consumption."
)


display(
    executive_summary_df
)